# Notebook 04b — Análisis topológico de parches (Cap. 16, Práctica 2)

Complementa al notebook `04_fragmentation.ipynb` (Julia) aplicando los predicados
espaciales DE-9IM enseñados en el **Cap. 16** del curso y la **Práctica 2 de
vectores**. Mientras Julia se encarga de las métricas geométricas
(área, perímetro, MSI), Python aporta la lógica topológica con `geopandas` y
`shapely`, que delegan en GEOS exactamente los mismos predicados que reconocen
PostGIS, ArcGIS y JTS.

Se evalúan dos predicados concretos sobre los parches reproyectados a EPSG:9377:

1. **`intersects(parche, frontera_AOI)`** → identifica parches truncados por
   el borde del área de estudio. Sus métricas de fragmentación son artefactos
   del recorte y deben reportarse como tales.
2. **`contains(parche, estación)`** → identifica parches que contienen
   estaciones de muestreo INVEMAR, lo que permite discutir la representatividad
   de la segmentación frente a los puntos de validación in situ.


In [ ]:
import sys, os
sys.path.insert(0, '../src')
from python.utils import (
    parches_borde, parches_con_punto, area_ha_9377, EPSG_NACIONAL,
)
import geopandas as gpd
from shapely.geometry import Point
import pandas as pd

base_dir = '../data/processed/samgeo'
out_dir  = '../outputs/tables'
os.makedirs(out_dir, exist_ok=True)
print(f'Sistema de referencia objetivo: {EPSG_NACIONAL}')


In [ ]:
# --- AOI y estaciones INVEMAR ---
aoi = gpd.read_file('../data/raw/cgsm_aoi.geojson')
if aoi.crs is None:
    aoi = aoi.set_crs('EPSG:4326')
aoi = aoi.to_crs(EPSG_NACIONAL)
print(f'AOI: {len(aoi)} polígono(s), CRS={aoi.crs}')

estaciones = gpd.GeoDataFrame({
    'estacion': ['Isla_Boqueron','Punta_Cerro','Punta_Chino','Rio_Sevilla',
                 'Cano_Palos','CP_Luna','CP_Aguas_Negras','Cano_Clarin'],
    'fuente':   ['INVEMAR']*5 + ['Complementaria']*3,
    'geometry': [Point(-74.298457,10.962255), Point(-74.283206,10.973076),
                 Point(-74.304827,10.912032), Point(-74.325228,10.880496),
                 Point(-74.471258,10.757558), Point(-74.56,10.87),
                 Point(-74.57,10.80), Point(-74.50,10.60)]
}, crs='EPSG:4326').to_crs(EPSG_NACIONAL)
print(f'Estaciones: {len(estaciones)}')


In [ ]:
# --- Predicados DE-9IM por período ---
filas = []
for nombre in ['degradacion', 'recuperacion', 'actual']:
    p = f'{base_dir}/manglar_{nombre}_9377.geojson'
    if not os.path.exists(p):
        print(f'  no encontrado: {p}'); continue

    gdf = gpd.read_file(p)
    gdf = area_ha_9377(gdf)
    # Filtro consistente con la metodología de fragmentación
    gdf = gdf[(gdf['area_ha'] >= 1.0) & (gdf['area_ha'] < 5000.0)].copy()

    gdf = parches_borde(gdf, aoi, tolerancia_m=30.0)
    gdf = parches_con_punto(gdf, estaciones)

    n_total       = len(gdf)
    n_borde       = int(gdf['borde'].sum())
    n_estacion    = int(gdf['con_estacion'].sum())
    area_borde    = float(gdf.loc[gdf['borde'], 'area_ha'].sum())
    area_estacion = float(gdf.loc[gdf['con_estacion'], 'area_ha'].sum())

    filas.append({
        'periodo': nombre,
        'parches': n_total,
        'parches_borde': n_borde,
        'pct_borde': round(100*n_borde/max(n_total,1), 1),
        'area_ha_borde': round(area_borde, 1),
        'parches_con_estacion': n_estacion,
        'estaciones_dentro': ', '.join(sorted(set(
            s for sub in gdf['estacion_dentro'].dropna() for s in sub.split(', ')
        ))),
    })

    # Guardar GeoJSON enriquecido para mapa
    out_geojson = f'{base_dir}/manglar_{nombre}_topologia.geojson'
    gdf[['area_ha','perim_m','borde','con_estacion','estacion_dentro','geometry']].to_file(
        out_geojson, driver='GeoJSON')
    print(f'  {nombre}: {n_total} parches, {n_borde} en borde, {n_estacion} con estación')

df_topo = pd.DataFrame(filas)
df_topo.to_csv(f'{out_dir}/parches_topologia.csv', index=False)
print('\nGuardado: outputs/tables/parches_topologia.csv')
print(df_topo.to_string(index=False))


## Lectura de los predicados

- **Parches en borde** (`intersects` con la frontera del AOI): cuando este
  porcentaje es alto, las métricas geométricas globales pierden poder
  comparativo, pues los parches están truncados por una decisión cartográfica
  externa. Conviene reportar las métricas en dos versiones —con y sin parches
  de borde— para que la fragmentación intra-AOI no quede contaminada por el
  recorte.
- **Parches con estación INVEMAR dentro** (`contains` con los 8 puntos de
  muestreo): permite verificar que las estaciones in situ —utilizadas para
  el análisis bfast en R y los z-scores en Python— efectivamente caen sobre
  parches segmentados como manglar. Es una validación cualitativa simple
  pero útil cuando, como en este proyecto, el F1 contra GMW es bajo y se
  necesita una segunda fuente de respaldo.


## Clasificación de estaciones (naturaleza + asociación al AOI)

Las 8 estaciones de muestreo no son homogéneas: las 4 INVEMAR-GBIF originales del
costado oriental (Isla_Boqueron, Punta_Cerro, Punta_Chino, Rio_Sevilla) son
**estaciones limnológicas sobre el agua** del sistema lagunar, mientras que
Cano_Palos (INVEMAR) y las 3 complementarias (CP_Luna, CP_Aguas_Negras, Cano_Clarin)
están sobre manglar. Esto se evidencia por el NDVI mediano histórico:
las limnológicas tienen NDVI < 0,3 y las de manglar NDVI > 0,4.

Además, solo Cano_Palos cae estrictamente **dentro del AOI acotado** (SFF + VPI).
Con buffer de 2 km, también quedan asociadas Cano_Clarin y Rio_Sevilla.
Las cinco restantes quedan fuera del AOI legal aunque pertenezcan al sistema
lagunar de la CGSM.


In [ ]:
from python.utils import (estaciones_geodataframe, etiquetar_estaciones_aoi,
                          EPSG_NACIONAL)

est = estaciones_geodataframe()
est = etiquetar_estaciones_aoi(est, aoi, buffer_m=2000.0)

# Cruce con NDVI mediano histórico
ndvi_hist = pd.read_csv('../outputs/tables/serie_temporal_ndvi_definitiva.csv')
ndvi_mediano = ndvi_hist.groupby('subzona')['ndvi'].median().round(3).to_dict()
est['ndvi_mediano'] = est['estacion'].map(ndvi_mediano)

df_est = pd.DataFrame(est.drop(columns='geometry'))
df_est['dist_a_aoi_m'] = df_est['dist_a_aoi_m'].round(0).astype(int)
df_est = df_est.sort_values(['naturaleza','fuente','estacion']).reset_index(drop=True)
df_est.to_csv('../outputs/tables/estaciones_clasificadas.csv', index=False)
print(df_est.to_string(index=False))


In [ ]:
# Resumen por naturaleza
resumen = (df_est.groupby('naturaleza')
           .agg(n=('estacion','count'),
                ndvi_medio=('ndvi_mediano','mean'),
                dist_media_km=('dist_a_aoi_m', lambda s: round(s.mean()/1000, 2)),
                n_asociadas=('asociada_aoi','sum'))
           .round(3))
print('Resumen por naturaleza:')
print(resumen)
print('\nGuardado: outputs/tables/estaciones_clasificadas.csv')


## Comparación dinámica del NDVI: manglar vs limnológicas

Con la clasificación anterior se puede separar la serie temporal en dos grupos y
verificar si la dinámica observada durante el evento La Niña 2020 difiere entre
estaciones de manglar y limnológicas. La hipótesis es que las estaciones de
manglar muestran una respuesta retardada al exceso hídrico (mortalidad gradual
del dosel) mientras que las limnológicas reflejan inmediatamente el cambio
del cuerpo de agua.


In [ ]:
# Comparación de series por grupo
ndvi_hist['fecha'] = pd.to_datetime(ndvi_hist['date'] if 'date' in ndvi_hist.columns else ndvi_hist['fecha'])
ndvi_hist = ndvi_hist.rename(columns={'subzona':'estacion'})
ndvi_hist = ndvi_hist.merge(df_est[['estacion','naturaleza']], on='estacion', how='left')

# Promedio mensual por grupo
ndvi_hist['anio_mes'] = ndvi_hist['fecha'].dt.to_period('M').dt.to_timestamp()
g = ndvi_hist.groupby(['anio_mes','naturaleza'])['ndvi'].mean().unstack()
g.tail(10)


In [ ]:
# Gráfico comparativo
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(12, 4.5))
if 'manglar' in g.columns:
    ax.plot(g.index, g['manglar'], color='#2ca02c', label='Manglar (n=4)', linewidth=1.5)
if 'limnologica' in g.columns:
    ax.plot(g.index, g['limnologica'], color='#1f77b4', label='Limnológica (n=4)', linewidth=1.5)
ax.axvspan(pd.Timestamp('2020-07-01'), pd.Timestamp('2020-12-31'),
           color='red', alpha=0.1, label='Degradación 2020')
ax.set_ylabel('NDVI promedio mensual')
ax.set_title('Dinámica NDVI por naturaleza de estación (CGSM 2018–2025)')
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
plt.savefig('../outputs/figures/ndvi_por_naturaleza.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: outputs/figures/ndvi_por_naturaleza.png')


In [2]:
import sys; sys.path.insert(0, '../src')
from python.utils import (
    area_ha_9377, parches_borde, parches_con_punto,
    estaciones_geodataframe, etiquetar_estaciones_aoi,
)
import geopandas as gpd
import pandas as pd

base = '../data/processed/samgeo_acotado'
aoi = gpd.read_file('../data/raw/cgsm_aoi_acotado_9377.geojson')
estaciones = estaciones_geodataframe().to_crs(9377)

filas = []
for nombre in ['degradacion', 'recuperacion', 'actual']:
    p = f'{base}/manglar_{nombre}_9377.geojson'
    gdf = gpd.read_file(p)
    gdf = area_ha_9377(gdf)
    gdf = gdf[(gdf['area_ha'] >= 1.0) & (gdf['area_ha'] < 5000.0)].copy()

    gdf = parches_borde(gdf, aoi, tolerancia_m=30.0)
    gdf = parches_con_punto(gdf, estaciones)

    n_total    = len(gdf)
    n_borde    = int(gdf['borde'].sum())
    n_estacion = int(gdf['con_estacion'].sum())
    area_borde = float(gdf.loc[gdf['borde'], 'area_ha'].sum())

    filas.append({
        'periodo': nombre,
        'parches': n_total,
        'parches_borde': n_borde,
        'pct_borde': round(100 * n_borde / max(n_total, 1), 1),
        'area_ha_borde': round(area_borde, 1),
        'parches_con_estacion': n_estacion,
        'estaciones_dentro': ', '.join(sorted(set(
            s for sub in gdf['estacion_dentro'].dropna() for s in sub.split(', ')
        ))),
    })

    # Guardar geojson enriquecido con topología
    out = f'{base}/manglar_{nombre}_topologia.geojson'
    gdf[['area_ha','perim_m','borde','con_estacion','estacion_dentro','geometry']].to_file(
        out, driver='GeoJSON')

df_topo = pd.DataFrame(filas)
df_topo.to_csv('../outputs/tables/parches_topologia_acotado.csv', index=False)
print(df_topo.to_string(index=False))
print('\nGuardado: outputs/tables/parches_topologia_acotado.csv')

     periodo  parches  parches_borde  pct_borde  area_ha_borde  parches_con_estacion estaciones_dentro
 degradacion       17              6       35.3         5104.3                     0                  
recuperacion       17              8       47.1         8325.7                     0                  
      actual       15              8       53.3         5237.9                     0                  

Guardado: outputs/tables/parches_topologia_acotado.csv
